# Surface-to-Air Missile (SAM) Intercept Simulation — v2 (Fixed)

**Fixes applied:**
1. `plt.savefig` output directory created before saving
2. Missile starts at `y=10m` so `ground_hit` event doesn't fire at `t=0`
3. Derivative PID state replaced with proper filtered derivative (`dxd = (err - xd) / TAU_D`)
4. Solver tolerances relaxed (`max_step=0.1`, `rtol=1e-4`, `atol=1e-6`) to prevent timeout

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os

In [ ]:
# ── Physical Parameters ───────────────────────────────────────────────────────
g    = 9.81
rho0 = 1.225
H    = 8500.0
Cd   = 0.3
A    = 0.05
m    = 300.0
N_pn = 4.0

# ── PID Gains ─────────────────────────────────────────────────────────────────
Kp, Ki, Kd = 3.5, 0.1, 0.8

# ── Derivative filter time constant ───────────────────────────────────────────
# FIX 3: Replaces the incorrect e_prev difference approach.
# dxd/dt = (err - xd) / TAU_D gives a proper continuous derivative estimate.
TAU_D = 0.05

# ── Acceleration Limit ────────────────────────────────────────────────────────
a_max = 30 * g

# ── Seeker Noise ──────────────────────────────────────────────────────────────
rng       = np.random.default_rng(42)
NOISE_STD = 0.0005

print('Parameters loaded.')

In [ ]:
# ── Maneuvering Target ────────────────────────────────────────────────────────
def target_state(t):
    x0, y0 = 15000.0, 5000.0
    vx = -250.0
    if t > 5.0:
        tau = t - 5.0
        vy  = 80.0 * np.sin(2 * np.pi * 0.4 * tau)
        dy  = (80.0 / (2 * np.pi * 0.4)) * (1 - np.cos(2 * np.pi * 0.4 * tau))
        y   = np.clip(y0 + dy, 500.0, 9000.0)
    else:
        vy, y = 0.0, y0
    return np.array([x0 + vx * t, y, vx, vy])

# ── Atmosphere ────────────────────────────────────────────────────────────────
def atm(y):
    return rho0 * np.exp(-max(y, 0.0) / H)

# ── ODE: State = [x, y, vx, vy, xi_pid, xd_filtered] ────────────────────────
def eom(t, state):
    x, y, vx, vy, xi, xd = state

    # Relative geometry
    tgt      = target_state(t)
    Rx, Ry   = tgt[0] - x,  tgt[1] - y
    R        = np.hypot(Rx, Ry) + 1e-6
    Rvx, Rvy = tgt[2] - vx, tgt[3] - vy

    # LOS rate (true) + seeker noise
    sigma_dot      = (Rx * Rvy - Ry * Rvx) / R**2
    sigma_dot_meas = sigma_dot + rng.normal(0, NOISE_STD)

    # Closing speed
    Vc = -(Rx * Rvx + Ry * Rvy) / R

    # PN guidance law
    a_pn = N_pn * Vc * sigma_dot_meas

    # PID autopilot (continuous ODE form)
    err  = -sigma_dot_meas          # drive LOS rate to zero
    dxi  = err                      # d(xi)/dt  — integral state
    dxd  = (err - xd) / TAU_D      # d(xd)/dt  — FIX 3: filtered derivative
    a_pid = Kp * err + Ki * xi + Kd * xd

    # Blended command with 30g clamp
    a_cmd = np.clip(a_pn + 0.2 * a_pid, -a_max, a_max)

    # Drag
    V    = np.hypot(vx, vy) + 1e-6
    Fd   = 0.5 * atm(y) * V**2 * Cd * A
    ax_d = -Fd * (vx / V) / m
    ay_d = -Fd * (vy / V) / m

    # Lateral command decomposition
    theta = np.arctan2(vy, vx)
    ax = ax_d + a_cmd * (-np.sin(theta))
    ay = ay_d + a_cmd * np.cos(theta) - g

    return [vx, vy, ax, ay, dxi, dxd]

print('Functions defined.')

In [ ]:
# ── Initial Conditions ────────────────────────────────────────────────────────
v0, theta0 = 600.0, np.radians(35)
state0 = [
    0.0, 10.0,                          # FIX 2: y=10m avoids ground_hit at t=0
    v0 * np.cos(theta0),
    v0 * np.sin(theta0),
    0.0, 0.0
]

# ── Ground Hit Event ──────────────────────────────────────────────────────────
def ground_hit(t, s): return s[1]
ground_hit.terminal  = True
ground_hit.direction = -1

# ── Integrate ─────────────────────────────────────────────────────────────────
# FIX 4: Relaxed tolerances prevent solver timeout
sol = solve_ivp(
    eom, [0, 60], state0,
    t_eval=np.linspace(0, 60, 3000),
    max_step=0.1, rtol=1e-4, atol=1e-6,
    events=ground_hit
)

print(f'Solver status: {sol.message}')

In [ ]:
# ── Results ───────────────────────────────────────────────────────────────────
t  = sol.t
mx, my = sol.y[0], sol.y[1]
tgt_arr = np.array([target_state(ti) for ti in t])
tgt_x, tgt_y = tgt_arr[:, 0], tgt_arr[:, 1]
miss  = np.hypot(mx - tgt_x, my - tgt_y)
i_int = np.argmin(miss)

print('=' * 50)
print('  SAM Intercept Simulation — v2 (Fixed)')
print('=' * 50)
print(f'  Intercept time  : {t[i_int]:.2f} s')
print(f'  Miss distance   : {miss[i_int]:.2f} m')
print(f'  Intercept point : ({mx[i_int]:.0f}, {my[i_int]:.0f}) m')
print(f'  Missile speed   : {np.hypot(sol.y[2][i_int], sol.y[3][i_int]):.1f} m/s')
print('=' * 50)

In [ ]:
# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('SAM Intercept v2 — PN + PID Autopilot + Maneuvering Target',
             fontsize=13, fontweight='bold')

ax = axes[0, 0]
ax.plot(mx/1e3, my/1e3, 'b-', lw=2, label='Missile')
ax.plot(tgt_x/1e3, tgt_y/1e3, 'r--', lw=2, label='Target (jinking)')
ax.scatter(mx[i_int]/1e3, my[i_int]/1e3, s=180, c='gold',
           marker='*', zorder=6, label=f'Intercept @ {t[i_int]:.1f}s')
ax.set(xlabel='Range (km)', ylabel='Altitude (km)', title='2D Trajectory')
ax.legend(); ax.grid(True)

ax = axes[0, 1]
ax.semilogy(t, miss, 'm-', lw=1.5)
ax.axvline(t[i_int], ls='--', color='gold', label=f'CPA = {miss[i_int]:.1f} m')
ax.set(xlabel='Time (s)', ylabel='Miss Distance (m, log)', title='Miss Distance vs Time')
ax.legend(); ax.grid(True)

ax = axes[1, 0]
speed = np.hypot(sol.y[2], sol.y[3])
ax.plot(t, speed, 'g-', lw=1.5)
ax.set(xlabel='Time (s)', ylabel='Speed (m/s)', title='Missile Speed')
ax.grid(True)

ax = axes[1, 1]
ax.plot(t, my/1e3,     'b-',  lw=1.5, label='Missile')
ax.plot(t, tgt_y/1e3,  'r--', lw=1.5, label='Target')
ax.set(xlabel='Time (s)', ylabel='Altitude (km)', title='Altitude Profile')
ax.legend(); ax.grid(True)

plt.tight_layout()

# FIX 1: Ensure output directory exists before saving
os.makedirs('outputs', exist_ok=True)
plt.savefig('outputs/sam_simulation_v2.png', dpi=150, bbox_inches='tight')
print('Plot saved to outputs/sam_simulation_v2.png')
plt.show()